# Fine-to-Coarse Sequential Emotion Recognition — Colab runner

RoBERTa-base → GoEmotions₂₇ → RoBERTa_G + UTE → MELD₇.

Runtime → Change runtime type → **GPU** (T4 is enough) before running anything.
Full pipeline (Stage I + M1–M4) is roughly 2–3 hours on a T4.

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Clone and install

In [ ]:
REPO_URL = "https://github.com/sainadth/fine-to-coarse-emotion-recognition.git"

import os
if not os.path.exists("fine-to-coarse-emotion-recognition"):
    !git clone $REPO_URL
%cd fine-to-coarse-emotion-recognition
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. (Optional) keep checkpoints on Drive

Colab sessions die; mounting Drive means a crashed run does not cost the Stage-I backbone.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/f2c/checkpoints" "/content/drive/MyDrive/f2c/results"
    os.environ["F2C_CHECKPOINT_DIR"] = "/content/drive/MyDrive/f2c/checkpoints"
    os.environ["F2C_RESULTS_DIR"] = "/content/drive/MyDrive/f2c/results"
print(os.environ.get("F2C_CHECKPOINT_DIR", "checkpoints/ (local)"))

## 3. Data

GoEmotions comes from the Hub; MELD is downloaded as the three official CSVs.

In [ ]:
!python scripts/prepare_data.py --dataset all

## 4. Smoke test (≈2 min)

Touches every code path before committing to a long run.

In [ ]:
!python scripts/train_stage2.py --config configs/smoke.yaml

## 5. Stage I — GoEmotions, 27 fine-grained classes

The 27-class head is discarded afterwards; only θ_G transfers.

In [ ]:
!python scripts/train_stage1.py --config configs/stage1.yaml

## 6. Stage II — the four required systems

M₁/M₂ classify utterances independently; M₃/M₄ add the utterance-level Transformer encoder.

In [ ]:
!python scripts/train_stage2.py --system M1
!python scripts/train_stage2.py --system M2

In [ ]:
!python scripts/train_stage2.py --system M3
!python scripts/train_stage2.py --system M4

## 7. (Optional) research extension

e_t = [h_t ; p_t^f]: the frozen Stage-I 27-class distribution concatenated into the UTE input.

In [ ]:
!python scripts/train_stage2.py --config configs/m5_extension_fine_logits.yaml

## 8. Tables and error analysis

In [ ]:
!python scripts/build_results_table.py
!python scripts/error_analysis.py

In [ ]:
import json, pathlib

for path in sorted(pathlib.Path(os.environ.get("F2C_RESULTS_DIR", "results")).glob("*/metrics.json")):
    payload = json.loads(path.read_text())
    if payload.get("stage") != "II":
        continue
    test = payload["test"]
    print(f"{payload['system']:>4}  acc={test['accuracy']:.4f}  "
          f"macro-F1={test['macro_f1']:.4f}  weighted-F1={test['weighted_f1']:.4f}")

## 9. Download the artifacts

In [ ]:
!zip -qr f2c_artifacts.zip results docs/tables
from google.colab import files
files.download("f2c_artifacts.zip")